# San Antonio Bicycle High Injury Network

This notebook uses the City's official 22 Bicycle High Injury Network corridors. It compares 2019–2023 with 2024 through Sept. 1, 2026, then screens the current period for candidate new corridors.

In [ ]:
from pathlib import Path
import zipfile
import requests
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
BOUNDARIES = ROOT / 'data' / 'boundaries'
OUT = ROOT / 'outputs'
BOUNDARIES.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

In [ ]:
# Load CRIS and make one row per qualifying bicyclist crash.
raw = pd.read_csv(RAW / 'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['Person Type'] == '3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['year'] = pd.to_numeric(target['Crash Year'], errors='coerce')
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
sa = target[(target['City'] == 'SAN ANTONIO') & target['year'].between(2019, 2026)].copy()
crashes = (sa.groupby('Crash ID', as_index=False).agg(year=('year','first'), latitude=('latitude','first'), longitude=('longitude','first'), deaths=('death','sum'), serious_injuries=('serious_injury','sum')))
geo = crashes.dropna(subset=['latitude','longitude'])
points = gpd.GeoDataFrame(geo, geometry=gpd.points_from_xy(geo['longitude'], geo['latitude']), crs=4326)
print('San Antonio qualifying crashes:', crashes['Crash ID'].nunique())

In [ ]:
# Use the City's official Bicycle HIN corridor layer.
hin_url = ('https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/'
           'SS4A_HIN_Dashboard_Data/FeatureServer/1/query?'
           'where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson')
hin_file = BOUNDARIES / 'bicycle_hin_corridors.geojson'
response = requests.get(hin_url, timeout=120)
response.raise_for_status()
hin_file.write_bytes(response.content)
hin = gpd.read_file(hin_file).to_crs(4326).rename(columns={'Name':'corridor'})
print('Official HIN corridors:', len(hin))

# Match points to existing corridors using a 150-foot buffer. This does not create corridors.
hin_buffer = hin.to_crs(2278).copy()
hin_buffer['geometry'] = hin_buffer.geometry.buffer(150)
matches = gpd.sjoin(points.to_crs(2278), hin_buffer[['bicycle_hin_id','corridor','Miles','geometry']], how='inner', predicate='within')
matches = matches.drop_duplicates(['Crash ID','bicycle_hin_id'])

def summarize(frame):
    return (frame.groupby(['bicycle_hin_id','corridor','Miles'], as_index=False)
        .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'), first_year=('year','min'), last_year=('year','max')))

baseline = summarize(matches[matches['year'].between(2019,2023)])
current = summarize(matches[matches['year'].between(2024,2026)])
baseline.to_csv(OUT / 'official_bicycle_hin_2019_2023.csv', index=False)
current.to_csv(OUT / 'official_bicycle_hin_2024_2026.csv', index=False)

for period, mask in {'2019_2023': crashes['year'].between(2019,2023), '2024_2026': crashes['year'].between(2024,2026)}.items():
    period_crashes = crashes[mask]
    period_matches = matches[matches['year'].between(2019,2023) if period == '2019_2023' else matches['year'].between(2024,2026)]
    matched_ids = period_matches['Crash ID'].drop_duplicates()
    outside = period_crashes[~period_crashes['Crash ID'].isin(matched_ids)]
    outside.to_csv(OUT / f'qualifying_crashes_outside_official_hin_{period}.csv', index=False)
    print(f'Qualifying crashes outside official HIN ({period}):', len(outside))

display(baseline.sort_values(['crashes','deaths'], ascending=False))
display(current.sort_values(['crashes','deaths'], ascending=False))

In [ ]:
# Candidate new corridors for 2024–Sept. 1, 2026.
# This uses connected crash-bearing street segments, not a fixed-radius crash cluster.
streets_dir = RAW / 'streets'
street_shp = streets_dir / 'Streets' / 'Streets.shp'
if not street_shp.exists():
    with zipfile.ZipFile(RAW / 'Streets.zip') as archive:
        archive.extractall(streets_dir)
roads = gpd.read_file(street_shp)
roads = roads.rename(columns={'CartID':'segmentid','MSAG_NAME':'road_label','FROM_STREE':'from_street','TO_STREET':'to_street','CoSARoadFu':'road_class'})
roads['length_miles'] = roads['LengthFeet'] / 5280
roads = roads[roads['length_miles'] > 0].copy()
current_points = points[points['year'].between(2024,2026)].copy()
current_matches = gpd.sjoin_nearest(current_points.to_crs(roads.crs), roads[['segmentid','road_label','from_street','to_street','road_class','length_miles','geometry']], how='left', distance_col='match_distance_ft')
current_matches = current_matches[current_matches['match_distance_ft'] <= 150].sort_values('match_distance_ft').drop_duplicates('Crash ID')

crash_segments = roads[roads['segmentid'].isin(current_matches['segmentid'])].copy().reset_index(drop=True)
parent = list(range(len(crash_segments)))
def find(item):
    while parent[item] != item:
        parent[item] = parent[parent[item]]
        item = parent[item]
    return item
def union(left, right):
    left_root, right_root = find(left), find(right)
    if left_root != right_root:
        parent[right_root] = left_root
for left_index, geometry in enumerate(crash_segments.geometry):
    for right_index in crash_segments.sindex.query(geometry.buffer(25), predicate='intersects'):
        union(left_index, int(right_index))
crash_segments['corridor_id'] = [find(i) for i in range(len(crash_segments))]
segment_to_corridor = dict(zip(crash_segments['segmentid'], crash_segments['corridor_id']))
current_matches['corridor_id'] = current_matches['segmentid'].map(segment_to_corridor)

def join_values(series):
    values = sorted({str(value).strip() for value in series.dropna() if str(value).strip() and str(value).strip().upper() != 'TBD'})
    return ', '.join(values)
candidate_corridors = (current_matches.groupby('corridor_id', as_index=False)
    .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'), first_year=('year','min'), last_year=('year','max'), roads=('road_label',join_values), from_streets=('from_street',join_values), to_streets=('to_street',join_values), segments=('segmentid','nunique'), max_match_distance_ft=('match_distance_ft','max')))
corridor_lengths = crash_segments.groupby('corridor_id', as_index=False)['length_miles'].sum()
candidate_corridors = candidate_corridors.merge(corridor_lengths, on='corridor_id', how='left')
candidate_corridors = candidate_corridors[candidate_corridors['crashes'] >= 2].sort_values(['crashes','deaths','serious_injuries'], ascending=False)
candidate_corridors.to_csv(OUT / 'candidate_new_hin_corridors_2024_2026.csv', index=False)
print('Candidate new corridors:', len(candidate_corridors))
display(candidate_corridors)

## Interpretation

The official tables show whether the City's 22 corridors remain active. The candidate table is an exploratory screen for new corridors: it groups only connected street segments that each received a current-period qualifying crash. It is not presented as the City's exact scoring model, which has not been published.